In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Celebal Week 5") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [3]:
from google.colab import files

uploaded = files.upload()

Saving ecommerce_sales_data.csv to ecommerce_sales_data.csv


In [4]:
df = spark.read.csv(
    "ecommerce_sales_data.csv",
    header=True,
    inferSchema=True
)

In [5]:
df.show(5)

+----------+--------+-----------+----------+----------------+------------+-------------+----------+------------+----------------+-----------------+------------+-------------+-------------+------------+---------------+----------------+-------------+-----------+----------------+
|      Date|Order ID|Customer ID|Product ID|Product Category|Product Name|Quantity Sold|Unit Price|Discount (%)|  Payment Method|Customer Location|Order Status|Shipping Cost|Profit Margin|Customer Age|Customer Gender|Customer Segment|Review Rating|Total Sales|Discounted Price|
+----------+--------+-----------+----------+----------------+------------+-------------+----------+------------+----------------+-----------------+------------+-------------+-------------+------------+---------------+----------------+-------------+-----------+----------------+
|2024-01-01|ORD00001|   CUST1654|    PROD23|        Clothing|     T-Shirt|            5|     28.75|          15|Cash on Delivery|      Los Angeles|   Completed|      

In [6]:
df.printSchema()

root
 |-- Date: date (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Product Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Quantity Sold: integer (nullable = true)
 |-- Unit Price: double (nullable = true)
 |-- Discount (%): integer (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Customer Location: string (nullable = true)
 |-- Order Status: string (nullable = true)
 |-- Shipping Cost: double (nullable = true)
 |-- Profit Margin: double (nullable = true)
 |-- Customer Age: integer (nullable = true)
 |-- Customer Gender: string (nullable = true)
 |-- Customer Segment: string (nullable = true)
 |-- Review Rating: integer (nullable = true)
 |-- Total Sales: double (nullable = true)
 |-- Discounted Price: double (nullable = true)



In [7]:
duplicate_removed_df = df.dropDuplicates(["Customer ID", "Date"])

print("Original Records :", df.count())
print("Records After Removing Duplicates :", duplicate_removed_df.count())

Original Records : 100
Records After Removing Duplicates : 100


In [8]:
duplicate_removed_df.show(5)

+----------+--------+-----------+----------+----------------+------------+-------------+----------+------------+--------------+-----------------+------------+-------------+-------------+------------+---------------+----------------+-------------+-----------+----------------+
|      Date|Order ID|Customer ID|Product ID|Product Category|Product Name|Quantity Sold|Unit Price|Discount (%)|Payment Method|Customer Location|Order Status|Shipping Cost|Profit Margin|Customer Age|Customer Gender|Customer Segment|Review Rating|Total Sales|Discounted Price|
+----------+--------+-----------+----------+----------------+------------+-------------+----------+------------+--------------+-----------------+------------+-------------+-------------+------------+---------------+----------------+-------------+-----------+----------------+
|2024-02-08|ORD00039|   CUST1006|    PROD42|        Clothing|Coffee Maker|            2|    112.99|          15|    Debit Card|      Los Angeles|   Cancelled|        19.71|

In [11]:
from pyspark.sql.functions import avg

filtered_sales = df.filter(df["Customer Location"] == "Los Angeles") \
                   .groupBy("Product Category") \
                   .agg(avg("Total Sales").alias("Average Sales"))

filtered_sales.show()

+----------------+-----------------+
|Product Category|    Average Sales|
+----------------+-----------------+
|  Home & Kitchen|           929.61|
|          Sports|          1595.72|
|     Electronics|992.9599999999999|
|        Clothing|308.0028571428571|
|          Beauty|          1135.48|
+----------------+-----------------+



In [12]:
from pyspark.sql.functions import col, when

df_null = df.withColumn(
    "Order Status",
    when(col("Order ID") == "ORD00005", None)
    .otherwise(col("Order Status"))
)

In [13]:
filled_df = df_null.na.fill({"Order Status": "Unknown"})

filled_df.select("Order ID", "Order Status").show(10)

+--------+------------+
|Order ID|Order Status|
+--------+------------+
|ORD00001|   Completed|
|ORD00002|   Cancelled|
|ORD00003|    Returned|
|ORD00004|   Cancelled|
|ORD00005|     Unknown|
|ORD00006|   Cancelled|
|ORD00007|   Cancelled|
|ORD00008|    Returned|
|ORD00009|    Returned|
|ORD00010|   Cancelled|
+--------+------------+
only showing top 10 rows


In [16]:
from pyspark.sql.functions import count

city_count = df.groupBy("Customer Location") \
               .agg(count("*").alias("Total Records")) \
               .filter(col("Total Records") > 20)

city_count.show()

+-----------------+-------------+
|Customer Location|Total Records|
+-----------------+-------------+
|      Los Angeles|           22|
|         New York|           22|
+-----------------+-------------+



In [17]:
new_df = df.drop("Discount (%)")

renamed_df = df.withColumnRenamed("Customer Age", "Age")

print("Original Columns:")
print(df.columns)

print("\nRenamed DataFrame Columns:")
print(renamed_df.columns)

Original Columns:
['Date', 'Order ID', 'Customer ID', 'Product ID', 'Product Category', 'Product Name', 'Quantity Sold', 'Unit Price', 'Discount (%)', 'Payment Method', 'Customer Location', 'Order Status', 'Shipping Cost', 'Profit Margin', 'Customer Age', 'Customer Gender', 'Customer Segment', 'Review Rating', 'Total Sales', 'Discounted Price']

Renamed DataFrame Columns:
['Date', 'Order ID', 'Customer ID', 'Product ID', 'Product Category', 'Product Name', 'Quantity Sold', 'Unit Price', 'Discount (%)', 'Payment Method', 'Customer Location', 'Order Status', 'Shipping Cost', 'Profit Margin', 'Age', 'Customer Gender', 'Customer Segment', 'Review Rating', 'Total Sales', 'Discounted Price']


In [18]:
from pyspark.sql.functions import when, col

df_subscription = df.withColumn(
    "Subscription",
    when(col("Customer Age") <= 30, "Premium")
    .otherwise("Standard")
)

filtered_df = df_subscription.filter(
    (col("Customer Age").between(18, 30)) &
    (col("Subscription") == "Premium")
)

filtered_df.select(
    "Customer ID",
    "Customer Age",
    "Subscription",
    "Product Category",
    "Total Sales"
).show()

+-----------+------------+------------+----------------+-----------+
|Customer ID|Customer Age|Subscription|Product Category|Total Sales|
+-----------+------------+------------+----------------+-----------+
|   CUST1025|          25|     Premium|     Electronics|     896.95|
|   CUST1759|          26|     Premium|  Home & Kitchen|    1186.05|
|   CUST1281|          20|     Premium|          Sports|     223.83|
|   CUST1250|          20|     Premium|          Sports|     984.56|
|   CUST1754|          25|     Premium|  Home & Kitchen|     103.14|
|   CUST1104|          24|     Premium|        Clothing|     158.75|
|   CUST1913|          26|     Premium|     Electronics|    1215.57|
|   CUST1223|          27|     Premium|        Clothing|     946.05|
|   CUST1616|          24|     Premium|          Sports|    1804.08|
|   CUST1665|          20|     Premium|          Sports|     563.32|
|   CUST1284|          24|     Premium|          Sports|     208.98|
|   CUST1825|          21|     Pre

In [19]:
clean_df = df.na.fill({"Total Sales": 0})

clean_df.groupBy("Product Category") \
        .sum("Total Sales") \
        .show()

+----------------+------------------+
|Product Category|  sum(Total Sales)|
+----------------+------------------+
|  Home & Kitchen|13858.380000000001|
|          Sports|19059.489999999998|
|     Electronics|          15271.19|
|        Clothing|14132.649999999998|
|          Beauty|14609.570000000002|
+----------------+------------------+



In [20]:
from pyspark.sql.functions import to_timestamp

timestamp_df = df.withColumn(
    "event_time",
    to_timestamp("Date", "yyyy-MM-dd")
).drop("Date")


In [22]:
timestamp_df.select("event_time").show(10)

+-------------------+
|         event_time|
+-------------------+
|2024-01-01 00:00:00|
|2024-01-02 00:00:00|
|2024-01-03 00:00:00|
|2024-01-04 00:00:00|
|2024-01-05 00:00:00|
|2024-01-06 00:00:00|
|2024-01-07 00:00:00|
|2024-01-08 00:00:00|
|2024-01-09 00:00:00|
|2024-01-10 00:00:00|
+-------------------+
only showing top 10 rows


In [23]:
timestamp_df.printSchema()

root
 |-- Order ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Product Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Quantity Sold: integer (nullable = true)
 |-- Unit Price: double (nullable = true)
 |-- Discount (%): integer (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Customer Location: string (nullable = true)
 |-- Order Status: string (nullable = true)
 |-- Shipping Cost: double (nullable = true)
 |-- Profit Margin: double (nullable = true)
 |-- Customer Age: integer (nullable = true)
 |-- Customer Gender: string (nullable = true)
 |-- Customer Segment: string (nullable = true)
 |-- Review Rating: integer (nullable = true)
 |-- Total Sales: double (nullable = true)
 |-- Discounted Price: double (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [24]:
df.groupBy("Product Category") \
  .sum("Total Sales") \
  .show()


+----------------+------------------+
|Product Category|  sum(Total Sales)|
+----------------+------------------+
|  Home & Kitchen|13858.380000000001|
|          Sports|19059.489999999998|
|     Electronics|          15271.19|
|        Clothing|14132.649999999998|
|          Beauty|14609.570000000002|
+----------------+------------------+



In [25]:
from pyspark.sql.functions import lit, when, col

df_user = df.withColumn("Email", lit("customer@example.com")) \
            .withColumn("Username", lit("customer"))

df_user = df_user.withColumn(
    "Email",
    when(col("Customer ID") == "CUST001", None)
    .otherwise(col("Email"))
)

df_user = df_user.withColumn(
    "Username",
    when(col("Customer ID") == "CUST002", "")
    .otherwise(col("Username"))
)

clean_user_df = df_user.filter(
    col("Email").isNotNull() &
    (col("Username") != "")
)

clean_user_df.select(
    "Customer ID",
    "Email",
    "Username"
).show()

+-----------+--------------------+--------+
|Customer ID|               Email|Username|
+-----------+--------------------+--------+
|   CUST1654|customer@example.com|customer|
|   CUST1114|customer@example.com|customer|
|   CUST1025|customer@example.com|customer|
|   CUST1759|customer@example.com|customer|
|   CUST1281|customer@example.com|customer|
|   CUST1250|customer@example.com|customer|
|   CUST1228|customer@example.com|customer|
|   CUST1142|customer@example.com|customer|
|   CUST1754|customer@example.com|customer|
|   CUST1104|customer@example.com|customer|
|   CUST1692|customer@example.com|customer|
|   CUST1758|customer@example.com|customer|
|   CUST1913|customer@example.com|customer|
|   CUST1558|customer@example.com|customer|
|   CUST1089|customer@example.com|customer|
|   CUST1604|customer@example.com|customer|
|   CUST1432|customer@example.com|customer|
|   CUST1032|customer@example.com|customer|
|   CUST1030|customer@example.com|customer|
|   CUST1095|customer@example.co

In [27]:
from pyspark.sql.functions import min, max, avg

df.agg(
    min("Unit Price").alias("Minimum Price"),
    max("Unit Price").alias("Maximum Price"),
    avg("Unit Price").alias("Average Price")
).show()

+-------------+-------------+------------------+
|Minimum Price|Maximum Price|     Average Price|
+-------------+-------------+------------------+
|        28.75|       492.28|283.79790000000014|
+-------------+-------------+------------------+



In [28]:
df.printSchema()


root
 |-- Date: date (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Product Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Quantity Sold: integer (nullable = true)
 |-- Unit Price: double (nullable = true)
 |-- Discount (%): integer (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Customer Location: string (nullable = true)
 |-- Order Status: string (nullable = true)
 |-- Shipping Cost: double (nullable = true)
 |-- Profit Margin: double (nullable = true)
 |-- Customer Age: integer (nullable = true)
 |-- Customer Gender: string (nullable = true)
 |-- Customer Segment: string (nullable = true)
 |-- Review Rating: integer (nullable = true)
 |-- Total Sales: double (nullable = true)
 |-- Discounted Price: double (nullable = true)



In [29]:
from pyspark.sql.functions import sum

pipeline_df = df.dropDuplicates()

pipeline_df = pipeline_df.na.fill({
    "Unit Price": 0
})

result_df = pipeline_df.groupBy("Product Category") \
                       .agg(sum("Total Sales").alias("Total Revenue"))

result_df.show()

+----------------+------------------+
|Product Category|     Total Revenue|
+----------------+------------------+
|  Home & Kitchen|          13858.38|
|          Sports|19059.489999999998|
|     Electronics|          15271.19|
|        Clothing|          14132.65|
|          Beauty|14609.570000000002|
+----------------+------------------+

